In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
# ==========================================
# 1. CARGA DE DATOS
# ==========================================
df = pd.read_csv("insurance.csv")

print("Primeras filas del dataset:")
print(df.head())

print("\nDimensiones iniciales del dataset:")
print(df.shape)


Primeras filas del dataset:
   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520

Dimensiones iniciales del dataset:
(1338, 7)


In [4]:
# ==========================================
# 2. LIMPIEZA DE DATOS
# ==========================================
print("\nValores faltantes por columna:")
print(df.isnull().sum())

print("\nNúmero de duplicados antes de eliminar:")
print(df.duplicated().sum())

df = df.drop_duplicates()

print("\nDimensiones después de eliminar duplicados:")
print(df.shape)



Valores faltantes por columna:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Número de duplicados antes de eliminar:
1

Dimensiones después de eliminar duplicados:
(1337, 7)


In [5]:
# ==========================================
# 3. VALIDACIÓN DE TIPOS DE DATOS
# ==========================================
df["age"] = df["age"].astype(int)
df["children"] = df["children"].astype(int)
df["bmi"] = df["bmi"].astype(float)
df["charges"] = df["charges"].astype(float)

df["sex"] = df["sex"].astype(str)
df["smoker"] = df["smoker"].astype(str)
df["region"] = df["region"].astype(str)


In [6]:
# ==========================================
# 4. TRATAMIENTO DE OUTLIERS
# ==========================================
def cap_outliers_iqr(dataframe, columns):
    df_out = dataframe.copy()

    for col in columns:
        Q1 = df_out[col].quantile(0.25)
        Q3 = df_out[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_limit = Q1 - 1.5 * IQR
        upper_limit = Q3 + 1.5 * IQR

        df_out[col] = np.where(df_out[col] < lower_limit, lower_limit, df_out[col])
        df_out[col] = np.where(df_out[col] > upper_limit, upper_limit, df_out[col])

    return df_out

df = cap_outliers_iqr(df, ["bmi", "charges"])

In [7]:
# ==========================================
# 5. FEATURE ENGINEERING
# ==========================================
df["is_parent"] = (df["children"] > 0).astype(int)
df["smoker_binary"] = (df["smoker"] == "yes").astype(int)
df["age_bmi"] = df["age"] * df["bmi"]
df["bmi_smoker"] = df["bmi"] * df["smoker_binary"]
df["age_smoker"] = df["age"] * df["smoker_binary"]

In [8]:
# ==========================================
# 6. TRANSFORMACIÓN LOGARÍTMICA
# ==========================================
df["charges_log"] = np.log1p(df["charges"])

In [9]:
# ==========================================
# 7. VARIABLES PREDICTORAS Y OBJETIVO
# ==========================================
X = df.drop(columns=["charges", "charges_log"])
y = df["charges_log"]

In [10]:
# ==========================================
# 8. VARIABLES NUMÉRICAS Y CATEGÓRICAS
# ==========================================
numeric_features = [
    "age",
    "bmi",
    "children",
    "is_parent",
    "smoker_binary",
    "age_bmi",
    "bmi_smoker",
    "age_smoker"
]

categorical_features = [
    "sex",
    "region",
    "smoker"
]

In [11]:
# ==========================================
# 9. PREPROCESAMIENTO
# ==========================================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [12]:
# ==========================================
# 10. FUNCIÓN PARA ENTRENAR Y EVALUAR
# ==========================================
def entrenar_y_evaluar(test_size, nombre_modelo):
    print("\n" + "="*50)
    print(f"{nombre_modelo}")
    print("="*50)

    # División de datos
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

    # Pipeline completo: preprocesamiento + modelo
    pipeline_model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ])

    # Entrenamiento
    pipeline_model.fit(X_train, y_train)

    # Predicción en escala logarítmica
    y_pred_log = pipeline_model.predict(X_test)

    # Regresar a escala original
    y_test_original = np.expm1(y_test)
    y_pred_original = np.expm1(y_pred_log)

    # Métricas
    rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_original))
    mae = mean_absolute_error(y_test_original, y_pred_original)
    r2 = r2_score(y_test_original, y_pred_original)

    print(f"División train/test: {int((1-test_size)*100)}/{int(test_size*100)}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")

    return pipeline_model, rmse, mae, r2

In [13]:
# ==========================================
# 11. ENTRENAMIENTO DE LOS DOS MODELOS
# ==========================================
modelo_70_30, rmse_70_30, mae_70_30, r2_70_30 = entrenar_y_evaluar(
    test_size=0.30,
    nombre_modelo="MODELO 1 - REGRESIÓN MULTILINEAL (70/30)"
)

modelo_80_20, rmse_80_20, mae_80_20, r2_80_20 = entrenar_y_evaluar(
    test_size=0.20,
    nombre_modelo="MODELO 2 - REGRESIÓN MULTILINEAL (80/20)"
)


MODELO 1 - REGRESIÓN MULTILINEAL (70/30)
División train/test: 70/30
RMSE: 5263.2531
MAE: 2751.9028
R²: 0.7451

MODELO 2 - REGRESIÓN MULTILINEAL (80/20)
División train/test: 80/20
RMSE: 4994.5458
MAE: 2697.2328
R²: 0.7865


In [14]:
# ==========================================
# 12. COMPARACIÓN DE RESULTADOS
# ==========================================
resultados = pd.DataFrame({
    "Modelo": ["Regresión 70/30", "Regresión 80/20"],
    "RMSE": [rmse_70_30, rmse_80_20],
    "MAE": [mae_70_30, mae_80_20],
    "R2": [r2_70_30, r2_80_20]
})

print("\n" + "="*50)
print("COMPARACIÓN DE MODELOS")
print("="*50)
print(resultados)


COMPARACIÓN DE MODELOS
            Modelo         RMSE          MAE        R2
0  Regresión 70/30  5263.253129  2751.902845  0.745132
1  Regresión 80/20  4994.545785  2697.232781  0.786524


In [15]:
# ==========================================
# 13. SELECCIÓN DEL MEJOR MODELO
# ==========================================
# Criterio: mayor R²
if r2_80_20 > r2_70_30:
    mejor_modelo = modelo_80_20
    mejor_nombre = "Modelo 2 - Regresión 80/20"
else:
    mejor_modelo = modelo_70_30
    mejor_nombre = "Modelo 1 - Regresión 70/30"

print("\nMejor modelo seleccionado:", mejor_nombre)


Mejor modelo seleccionado: Modelo 2 - Regresión 80/20


In [16]:
# ==========================================
# 14. SERIALIZACIÓN DEL MODELO
# ==========================================
joblib.dump(mejor_modelo, "model.pkl")
print("\nModelo guardado correctamente como 'model.pkl'")


Modelo guardado correctamente como 'model.pkl'


In [17]:
mejor_modelo

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'bmi', 'children',
                                                   'is_parent', 'smoker_binary',
                                                   'age_bmi', 'bmi_smoker',
                                                   'age_smoker']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['sex', 'region',
                                                   'smoker'])])),
                ('model', LinearRegression())])

In [18]:
import pickle
pickle.dump(mejor_modelo, open('model.pkl', 'wb'))